# API Smoke Test

Notebook này chỉ test 3 việc tối thiểu:

1. Teacher LLM nhận input bài báo ngắn và gán nhãn thử.
2. Student LLM 8B nhận input và trả lời thử.
3. Embedding model nhận text và trả vector thử.

Notebook đọc cấu hình từ `.env` và không in API key.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path
from typing import Any

import requests


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "finevent").exists():
            return candidate
    raise RuntimeError("Không tìm thấy project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)


def load_env(path: Path) -> None:
    try:
        from dotenv import load_dotenv
    except ImportError:
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    else:
        load_dotenv(path)


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Thiếu biến môi trường: {name}")
    return value


def chat_completion(
    *,
    base_url: str,
    api_key: str,
    model: str,
    prompt: str,
    max_tokens: int = 512,
) -> dict[str, Any]:
    url = f"{base_url.rstrip('/')}/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
        "max_tokens": max_tokens,
    }
    response = requests.post(url, headers=headers, json=payload, timeout=120)
    if response.status_code == 400:
        payload.pop("max_tokens", None)
        payload["max_completion_tokens"] = max_tokens
        response = requests.post(url, headers=headers, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()


def extract_message_text(payload: dict[str, Any]) -> str:
    choices = payload.get("choices") or []
    if not choices:
        return ""
    message = choices[0].get("message") or {}
    content = message.get("content") or ""
    reasoning_content = message.get("reasoning_content") or ""
    if content:
        return str(content).strip()
    return str(reasoning_content).strip()


def embedding_request(*, base_url: str, api_key: str, model: str, texts: list[str]) -> dict[str, Any]:
    url = f"{base_url.rstrip('/')}/embeddings"
    response = requests.post(
        url,
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={"model": model, "input": texts},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()


load_env(PROJECT_ROOT / ".env")
print(json.dumps({"project_root": str(PROJECT_ROOT), "env_loaded": True}, ensure_ascii=False, indent=2))

In [ ]:
# Test 1: Teacher LLM gán nhãn thử
teacher_base_url = os.getenv("TEACHER_LLM_BASE_URL") or "https://api.openai.com/v1"
teacher_api_key = os.getenv("OPENAI_API_KEY") or os.getenv("TEACHER_LLM_API_KEY")
if not teacher_api_key:
    raise RuntimeError("Thiếu OPENAI_API_KEY hoặc TEACHER_LLM_API_KEY")
teacher_model = require_env("TEACHER_LLM_MODEL")

teacher_input = "HPG công bố kế hoạch mở rộng nhà máy thép tại Dung Quất để tăng công suất sản xuất."
teacher_prompt = f"""
Bạn là teacher LLM cho bài toán trích xuất sự kiện tài chính tiếng Việt.
Hãy gán nhãn thử cho văn bản sau và chỉ trả về JSON ngắn.

Văn bản: {teacher_input}

Schema: {{"ticker": string|null, "event_type": string, "impact_direction": "positive|negative|neutral", "evidence": string}}
""".strip()

started_at = time.perf_counter()
teacher_payload = chat_completion(
    base_url=teacher_base_url,
    api_key=teacher_api_key,
    model=teacher_model,
    prompt=teacher_prompt,
    max_tokens=512,
)
teacher_output = extract_message_text(teacher_payload)

print(json.dumps({
    "api": "teacher_llm",
    "model": teacher_model,
    "latency_seconds": round(time.perf_counter() - started_at, 3),
    "input": teacher_input,
    "output": teacher_output,
}, ensure_ascii=False, indent=2))

In [ ]:
# Test 2: Student LLM 8B trả lời thử
student_base_url = require_env("STUDENT_LLM_BASE_URL")
student_api_key = require_env("STUDENT_LLM_API_KEY")
student_model = require_env("STUDENT_LLM_MODEL")

student_input = "VCB bổ nhiệm tổng giám đốc mới từ tháng 7."
student_prompt = f"""
/no_think
Bạn là student LLM 8B. Hãy đọc văn bản và trả lời ngắn gọn bằng JSON.

Văn bản: {student_input}

Schema: {{"ticker": string|null, "event_type": string, "short_answer": string}}
""".strip()

started_at = time.perf_counter()
student_payload = chat_completion(
    base_url=student_base_url,
    api_key=student_api_key,
    model=student_model,
    prompt=student_prompt,
    max_tokens=512,
)
student_output = extract_message_text(student_payload)

print(json.dumps({
    "api": "student_llm",
    "model": student_model,
    "latency_seconds": round(time.perf_counter() - started_at, 3),
    "input": student_input,
    "output": student_output,
}, ensure_ascii=False, indent=2))

In [ ]:
# Test 3: Embedding model trả vector thử
embedding_base_url = require_env("EMBEDDING_BASE_URL")
embedding_api_key = os.getenv("EMBEDDING_API_KEY") or os.getenv("STUDENT_LLM_API_KEY")
if not embedding_api_key:
    raise RuntimeError("Thiếu EMBEDDING_API_KEY hoặc STUDENT_LLM_API_KEY")
embedding_model = require_env("EMBEDDING_MODEL")

embedding_texts = [
    "HPG mở rộng nhà máy thép tại Dung Quất.",
    "VCB bổ nhiệm tổng giám đốc mới.",
]

started_at = time.perf_counter()
embedding_payload = embedding_request(
    base_url=embedding_base_url,
    api_key=embedding_api_key,
    model=embedding_model,
    texts=embedding_texts,
)
embedding_data = embedding_payload.get("data") or []
vectors = [item.get("embedding") for item in embedding_data]
if not vectors or not vectors[0]:
    raise RuntimeError("Embedding API không trả vector.")

print(json.dumps({
    "api": "embedding",
    "model": embedding_model,
    "latency_seconds": round(time.perf_counter() - started_at, 3),
    "input_count": len(embedding_texts),
    "vector_count": len(vectors),
    "dimension": len(vectors[0]),
    "first_vector_preview": [round(float(value), 6) for value in vectors[0][:10]],
}, ensure_ascii=False, indent=2))